<a href="https://colab.research.google.com/github/roopaam/MB-Proxy/blob/main/PEP_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
import warnings
import ssl
import urllib.request
from io import StringIO
import certifi
warnings.filterwarnings('ignore')


# ==========================================
# SSL CERTIFICATE FIX
# ==========================================

def download_with_ssl_bypass(url: str) -> str:
    """Download data from URL with proper SSL certificate handling."""

    # Method 1: Try with certifi certificates
    try:
        print("  Attempting with certifi SSL certificates...")
        context = ssl.create_default_context(cafile=certifi.where())
        with urllib.request.urlopen(url, context=context) as response:
            return response.read().decode('utf-8')
    except Exception as e:
        print(f"    Failed: {str(e)[:50]}")

    # Method 2: Try with unverified context (less secure but works)
    try:
        print("  Attempting with unverified SSL context...")
        context = ssl._create_unverified_context()
        with urllib.request.urlopen(url, context=context) as response:
            return response.read().decode('utf-8')
    except Exception as e:
        print(f"    Failed: {str(e)[:50]}")

    # Method 3: Use alternative source (GitHub mirror)
    try:
        print("  Attempting GitHub mirror...")
        alt_url = "https://raw.githubusercontent.com/uci-ml-repo/uci-ml-repo.github.io/master/datasets/adult/adult.data"
        context = ssl._create_unverified_context()
        with urllib.request.urlopen(alt_url, context=context) as response:
            return response.read().decode('utf-8')
    except Exception as e:
        print(f"    Failed: {str(e)[:50]}")

    return None


# ==========================================
# CONDITIONAL MUTUAL INFORMATION COMPUTATION
# ==========================================

def conditional_mutual_information(y: np.ndarray, p: np.ndarray, s: np.ndarray) -> float:
    """
    Estimate I(Y; P | S) in nats using empirical plug-in probabilities.

    Formula:
        I(Y;P|S) = sum_s p(s) sum_y sum_p p(y,p|s) log( p(y,p|s) / (p(y|s) p(p|s)) )

    Args:
        y: Target variable (1D integer array)
        p: Proxy variable (1D integer array)
        s: Sensitive/protected attribute (1D integer array)

    Returns:
        Conditional mutual information value in nats (using natural logarithm)
    """
    # Ensure all inputs are 1D integer arrays
    y = np.asarray(y, dtype=int).flatten()
    p = np.asarray(p, dtype=int).flatten()
    s = np.asarray(s, dtype=int).flatten()

    # Remove rows with NaN or invalid values
    valid_idx = ~(np.isnan(y) | np.isnan(p) | np.isnan(s))
    y = y[valid_idx]
    p = p[valid_idx]
    s = s[valid_idx]

    if len(y) == 0:
        return 0.0

    n = len(y)
    cmi = 0.0

    # Get unique values
    s_values = np.unique(s)

    for s_val in s_values:
        # Get indices where S == s_val
        mask_s = (s == s_val)
        p_s = np.sum(mask_s) / n  # P(s)

        if p_s == 0 or np.sum(mask_s) == 0:
            continue

        # Subset data for this value of S
        y_s = y[mask_s]
        p_s_subset = p[mask_s]
        n_s = np.sum(mask_s)

        # Get unique values for Y and P given S
        y_values = np.unique(y_s)
        p_values = np.unique(p_s_subset)

        # Compute conditional MI for this S value
        for y_val in y_values:
            for p_val in p_values:
                # Count co-occurrences of y and p given s
                mask_y = (y == y_val)
                mask_p = (p == p_val)
                mask_yp_s = mask_y & mask_p & mask_s

                count_yp_s = np.sum(mask_yp_s)

                if count_yp_s == 0:
                    continue

                # P(y, p | s)
                P_yp_given_s = count_yp_s / n_s

                # P(y | s)
                count_y_s = np.sum(mask_y & mask_s)
                P_y_given_s = count_y_s / n_s if count_y_s > 0 else 0

                # P(p | s)
                count_p_s = np.sum(mask_p & mask_s)
                P_p_given_s = count_p_s / n_s if count_p_s > 0 else 0

                # Avoid log(0): only compute if probabilities are positive
                if P_y_given_s > 0 and P_p_given_s > 0 and P_yp_given_s > 0:
                    # Natural logarithm to get nats
                    ratio = P_yp_given_s / (P_y_given_s * P_p_given_s)
                    cmi += p_s * P_yp_given_s * np.log(ratio)

    return max(0.0, cmi)


# ==========================================
# PREPROCESSING FOR PEP
# ==========================================

def preprocess_for_pep(
    df: pd.DataFrame,
    target_col: str,
    sensitive_col: str,
    continuous_cols: list = None,
    n_bins: int = 5
) -> pd.DataFrame:
    """
    Preprocess dataframe for PEP estimation.

    - Discretizes continuous variables into quantile bins.
    - Label-encodes categorical variables.
    - Ensures all variables are integer-type suitable for CMI computation.

    Args:
        df: Input dataframe
        target_col: Name of target column
        sensitive_col: Name of protected attribute column
        continuous_cols: List of continuous column names to discretize
        n_bins: Number of quantile bins for discretization

    Returns:
        Preprocessed dataframe with discretized/encoded columns
    """
    df_proc = df.copy()

    if continuous_cols is None:
        continuous_cols = []

    # Discretize continuous variables
    for col in continuous_cols:
        if col not in df_proc.columns:
            continue

        try:
            # Try quantile-based discretization
            df_proc[col] = pd.qcut(
                df_proc[col],
                q=n_bins,
                labels=False,
                duplicates='drop'
            )
        except Exception as e:
            print(f"  Warning: Could not discretize {col} with qcut: {str(e)[:50]}")
            # Fall back to label encoding
            try:
                le = LabelEncoder()
                df_proc[col] = le.fit_transform(df_proc[col].astype(str))
            except Exception as e2:
                print(f"  Warning: Could not encode {col}: {str(e2)[:50]}")

    # Label-encode categorical (object) columns
    for col in df_proc.columns:
        if col in [target_col, sensitive_col]:
            continue

        if df_proc[col].dtype == 'object':
            try:
                le = LabelEncoder()
                df_proc[col] = le.fit_transform(df_proc[col].astype(str))
            except Exception as e:
                print(f"  Warning: Could not encode categorical column {col}: {str(e)[:50]}")

    # Ensure target and sensitive attribute are integer
    if df_proc[target_col].dtype == 'object':
        le = LabelEncoder()
        df_proc[target_col] = le.fit_transform(df_proc[target_col].astype(str))
    else:
        df_proc[target_col] = df_proc[target_col].astype(int)

    if df_proc[sensitive_col].dtype == 'object':
        le = LabelEncoder()
        df_proc[sensitive_col] = le.fit_transform(df_proc[sensitive_col].astype(str))
    else:
        df_proc[sensitive_col] = df_proc[sensitive_col].astype(int)

    return df_proc


# ==========================================
# PEP COMPUTATION FOR PROXIES
# ==========================================

def compute_pep_for_proxies(
    df: pd.DataFrame,
    target_col: str,
    sensitive_col: str,
    proxy_cols: list,
    continuous_cols: list = None,
    dataset_name: str = ""
) -> pd.DataFrame:
    """
    Compute PEP I(Y; P | S) for a set of proxy variables.

    Args:
        df: Input dataframe
        target_col: Name of target column
        sensitive_col: Name of protected attribute column
        proxy_cols: List of proxy variable column names
        continuous_cols: List of continuous columns to discretize
        dataset_name: Name of dataset for reporting

    Returns:
        DataFrame with columns:
        [dataset, proxy_variable, protected_attribute, target, pep_value_nats]
    """

    print(f"\n[Preprocessing {dataset_name} for PEP computation]")

    if continuous_cols is None:
        continuous_cols = []

    # Preprocess
    df_proc = preprocess_for_pep(
        df,
        target_col,
        sensitive_col,
        continuous_cols=continuous_cols
    )

    print(f"  Dataset shape after preprocessing: {df_proc.shape}")

    # Extract target and sensitive attribute
    y = df_proc[target_col].values
    s = df_proc[sensitive_col].values

    # Compute PEP for each proxy
    rows = []

    for proxy_col in proxy_cols:
        if proxy_col not in df_proc.columns:
            print(f"  Warning: Proxy column '{proxy_col}' not found. Skipping.")
            continue

        p = df_proc[proxy_col].values

        # Check for sufficient unique values
        n_unique_p = len(np.unique(p))
        if n_unique_p <= 1:
            print(f"  Warning: Proxy '{proxy_col}' has only {n_unique_p} unique value(s). PEP set to 0.")
            pep_value = 0.0
        else:
            print(f"  Computing PEP for '{proxy_col}'...", end=' ', flush=True)
            pep_value = conditional_mutual_information(y, p, s)
            print(f"PEP = {pep_value:.4f} nats")

        rows.append({
            "dataset": dataset_name,
            "proxy_variable": proxy_col,
            "protected_attribute": sensitive_col,
            "target": target_col,
            "pep_value_nats": pep_value,
        })

    result_df = pd.DataFrame(rows)
    return result_df


# ==========================================
# DATA LOADING - ADULT WITH MULTIPLE METHODS
# ==========================================

def load_adult_data_from_url() -> pd.DataFrame:
    """Load Adult dataset from UCI ML Repository with SSL bypass."""
    print("\n[Loading UCI Adult Dataset from URL]")

    columns = [
        "age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
        "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss",
        "hours-per-week", "native-country", "income",
    ]

    urls = [
        "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
        "https://raw.githubusercontent.com/uci-ml-repo/uci-ml-repo.github.io/master/datasets/adult/adult.data",
        "https://www.openml.org/data/get_csv/1595261/adult.csv",
    ]

    data_content = None

    for url in urls:
        print(f"  Trying: {url.split('/')[-1]}...", end=' ', flush=True)
        try:
            data_content = download_with_ssl_bypass(url)
            if data_content:
                print("✓")
                break
            else:
                print("✗")
        except Exception as e:
            print(f"✗ ({str(e)[:30]})")
            continue

    if data_content is None:
        raise ConnectionError("Could not download from any source. Please use local file.")

    # Parse the CSV data
    try:
        df = pd.read_csv(
            StringIO(data_content),
            names=columns,
            skipinitialspace=True,
            na_values=["?"],
        )
    except Exception as e:
        # Try without names if the above fails
        df = pd.read_csv(
            StringIO(data_content),
            skipinitialspace=True,
            na_values=["?"],
        )
        df.columns = columns

    print(f"  Original shape: {df.shape}")

    # Drop rows with missing values
    df = df.dropna().copy()
    print(f"  After dropping NaN: {df.shape}")

    # Drop fnlwgt (sampling weight)
    df = df.drop(columns=["fnlwgt", "education", "native-country"], errors='ignore')

    # Binarize target
    df["income"] = (df["income"].str.strip() == ">50K").astype(int)

    print(f"  Final shape: {df.shape}")

    return df


def load_adult_data_from_local() -> pd.DataFrame:
    """Load Adult dataset from local file."""
    print("\n[Loading Adult Dataset from Local File]")

    columns = [
        "age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
        "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss",
        "hours-per-week", "native-country", "income",
    ]

    # Try multiple possible filenames
    possible_files = ["adult.csv", "adult.data", "adult.txt"]

    for filename in possible_files:
        try:
            print(f"  Trying: {filename}...", end=' ', flush=True)
            df = pd.read_csv(
                filename,
                names=columns,
                skipinitialspace=True,
                na_values=["?"],
            )
            print("✓")
            print(f"  Original shape: {df.shape}")

            # Drop rows with missing values
            df = df.dropna().copy()
            print(f"  After dropping NaN: {df.shape}")

            # Drop fnlwgt (sampling weight)
            df = df.drop(columns=["fnlwgt", "education", "native-country"], errors='ignore')

            # Binarize target
            df["income"] = (df["income"].str.strip() == ">50K").astype(int)

            print(f"  Final shape: {df.shape}")

            return df
        except FileNotFoundError:
            print("✗")
            continue

    return None


def load_adult_data() -> pd.DataFrame:
    """Load Adult dataset with multiple fallback methods."""

    # Try loading from local file first
    df = load_adult_data_from_local()
    if df is not None:
        return df

    # Try loading from URL
    try:
        return load_adult_data_from_url()
    except Exception as e:
        print(f"\n✗ Could not load Adult dataset: {str(e)[:100]}")
        print("\nAlternative solutions:")
        print("  1. Download manually from: https://archive.ics.uci.edu/ml/datasets/adult")
        print("  2. Save the file as 'adult.csv' in the current directory")
        print("  3. Run this script again")
        print("\nOr use this command:")
        print("  wget https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data -O adult.csv")
        raise


# ==========================================
# DATA LOADING - ACSINCOME
# ==========================================

def load_acsincome_data(sample_size: int = 50000) -> pd.DataFrame:
    """Load and preprocess ACSIncome dataset from folktables."""
    print("\n[Loading ACSIncome Dataset]")

    try:
        from folktables import ACSDataSource, ACSIncome
    except ImportError:
        raise ImportError("Please install folktables: pip install folktables")

    data_source = ACSDataSource(
        survey_year="2018",
        horizon="1-Year",
        survey="person",
    )

    print("  Downloading ACS data for CA...")
    acs_data = data_source.get_data(states=["CA"], download=True)
    print(f"  Downloaded shape: {acs_data.shape}")

    features, labels, groups = ACSIncome.df_to_numpy(acs_data)
    feature_names = ACSIncome.features

    df = pd.DataFrame(features, columns=feature_names)
    df["income"] = labels
    df["sex"] = groups

    print(f"  Original shape: {df.shape}")

    df = df.dropna()
    print(f"  After dropping NaN: {df.shape}")

    if sample_size is not None and len(df) > sample_size:
        print(f"  Sampling {sample_size} records...")
        df = df.sample(n=sample_size, random_state=42)
        print(f"  Sampled shape: {df.shape}")

    return df


# ==========================================
# LATEX TABLE GENERATION
# ==========================================

def generate_latex_table(
    df: pd.DataFrame,
    save_path: str = "retained_proxy_pep_values.tex"
) -> None:
    """
    Generate LaTeX table suitable for inclusion in a paper.

    Args:
        df: DataFrame with columns: dataset, proxy_variable, pep_value_nats,
                                     pc_stable_stability, hc_bdeu_stability
        save_path: Output file path
    """

    with open(save_path, 'w') as f:
        f.write("\\begin{table}[h]\n")
        f.write("\\centering\n")
        f.write("\\caption{PEP values and MB selection stability for representative retained proxy variables. "
                "PEP is estimated as $I(Y;P\\mid S)$ in nats.}\n")
        f.write("\\label{tab:pep-stability}\n")
        f.write("\\begin{tabular}{llrrrr}\n")
        f.write("\\toprule\n")
        f.write("Dataset & Proxy & PEP $I(Y;P\\mid S)$ & PC-Stable & HC-BDeu \\\\\n")
        f.write("\\midrule\n")

        for _, row in df.iterrows():
            dataset = row["dataset"]
            proxy = row["proxy_variable"]
            pep = row["pep_value_nats"]

            pc_stable = row.get("pc_stable_stability", np.nan)
            hc_bdeu = row.get("hc_bdeu_stability", np.nan)

            # Format stability values
            pc_str = f"{pc_stable:.3f}" if not np.isnan(pc_stable) else "---"
            hc_str = f"{hc_bdeu:.3f}" if not np.isnan(hc_bdeu) else "---"

            f.write(f"{dataset} & {proxy} & {pep:.4f} & {pc_str} & {hc_str} \\\\\n")

        f.write("\\bottomrule\n")
        f.write("\\end{tabular}\n")
        f.write("\\end{table}\n")

    print(f"✓ Saved LaTeX table: {save_path}")


# ==========================================
# PLOTTING
# ==========================================

def plot_pep_values(
    df: pd.DataFrame,
    save_path: str = "retained_proxy_pep_values.png"
) -> None:
    """
    Create horizontal bar chart of PEP values.

    Args:
        df: DataFrame with columns: dataset, proxy_variable, pep_value_nats
        save_path: Output file path
    """

    # Sort by dataset then by descending PEP value
    df_sorted = df.sort_values(
        by=["dataset", "pep_value_nats"],
        ascending=[True, False]
    ).reset_index(drop=True)

    # Create labels combining dataset and proxy
    df_sorted["label"] = df_sorted["dataset"] + " - " + df_sorted["proxy_variable"]

    fig, ax = plt.subplots(figsize=(10, max(6, len(df_sorted) * 0.25)))

    # Color by dataset
    datasets = df_sorted["dataset"].unique()
    colors = {'Adult': '#1f77b4', 'ACSIncome': '#ff7f0e'}
    bar_colors = [colors.get(ds, '#2ca02c') for ds in df_sorted["dataset"]]

    y_pos = np.arange(len(df_sorted))
    ax.barh(y_pos, df_sorted["pep_value_nats"], color=bar_colors, alpha=0.8)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(df_sorted["label"], fontsize=10)
    ax.set_xlabel("PEP: I(Y; P | S) in nats", fontweight='bold', fontsize=12)
    ax.set_title("Proxy Entanglement Penalty (PEP) for Retained Proxy Variables",
                 fontweight='bold', fontsize=14)

    ax.grid(True, alpha=0.3, axis='x')
    ax.invert_yaxis()

    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=colors['Adult'], alpha=0.8, label='Adult'),
        Patch(facecolor=colors['ACSIncome'], alpha=0.8, label='ACSIncome')
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved plot: {save_path}")
    plt.close()


# ==========================================
# MAIN EXECUTION
# ==========================================

if __name__ == "__main__":
    print("="*80)
    print("PROXY ENTANGLEMENT PENALTY (PEP) COMPUTATION")
    print("UCI Adult and ACSIncome Datasets")
    print("="*80)

    all_results = []

    # ==========================================
    # 1. UCI ADULT
    # ==========================================

    print("\n" + "="*80)
    print("PART 1: UCI ADULT DATASET")
    print("="*80)

    try:
        df_adult = load_adult_data()

        adult_continuous_cols = [
            "age", "capital-gain", "capital-loss", "hours-per-week", "education-num"
        ]

        adult_proxy_cols = [
            "relationship", "marital-status", "occupation", "hours-per-week", "education-num"
        ]

        pep_adult = compute_pep_for_proxies(
            df_adult,
            target_col="income",
            sensitive_col="sex",
            proxy_cols=adult_proxy_cols,
            continuous_cols=adult_continuous_cols,
            dataset_name="Adult"
        )

        print(f"\n✓ Computed PEP for {len(pep_adult)} Adult proxies")
        all_results.append(pep_adult)

    except Exception as e:
        print(f"\n✗ Error processing Adult dataset: {e}")

    # ==========================================
    # 2. ACSINCOME
    # ==========================================

    print("\n" + "="*80)
    print("PART 2: ACSINCOME DATASET")
    print("="*80)

    try:
        df_acs = load_acsincome_data(sample_size=50000)

        acs_continuous_cols = ["AGEP", "WKHP", "SCHL"]

        acs_proxy_cols = ["OCCP", "SCHL", "WKHP", "RELP", "AGEP", "COW", "RAC1P"]

        pep_acs = compute_pep_for_proxies(
            df_acs,
            target_col="income",
            sensitive_col="sex",
            proxy_cols=acs_proxy_cols,
            continuous_cols=acs_continuous_cols,
            dataset_name="ACSIncome"
        )

        print(f"\n✓ Computed PEP for {len(pep_acs)} ACSIncome proxies")
        all_results.append(pep_acs)

    except Exception as e:
        print(f"\n✗ Error processing ACSIncome dataset: {e}")

    # ==========================================
    # 3. COMBINE AND ADD STABILITY VALUES
    # ==========================================

    if all_results:
        print("\n" + "="*80)
        print("COMBINING RESULTS")
        print("="*80)

        pep_combined = pd.concat(all_results, ignore_index=True)

        # Add stability values
        adult_stability = {
            "relationship": {"pc_stable": 0.913, "hc_bdeu": 1.000},
            "occupation": {"pc_stable": 0.920, "hc_bdeu": 0.380},
            "hours-per-week": {"pc_stable": 0.927, "hc_bdeu": 0.520},
            "education-num": {"pc_stable": 0.920, "hc_bdeu": 0.920},
            "marital-status": {"pc_stable": 0.920, "hc_bdeu": 0.113}
        }

        acs_stability = {
            "OCCP": {"pc_stable": 0.980, "hc_bdeu": 1.000},
            "SCHL": {"pc_stable": 0.980, "hc_bdeu": 1.000},
            "WKHP": {"pc_stable": 0.973, "hc_bdeu": 1.000},
            "RELP": {"pc_stable": 0.980, "hc_bdeu": 0.980},
            "AGEP": {"pc_stable": 0.980, "hc_bdeu": 0.893},
            "COW": {"pc_stable": 0.980, "hc_bdeu": 0.553},
            "RAC1P": {"pc_stable": 0.980, "hc_bdeu": 0.040}
        }

        # Initialize stability columns
        pep_combined["pc_stable_stability"] = np.nan
        pep_combined["hc_bdeu_stability"] = np.nan

        # Fill in Adult stability
        for idx, row in pep_combined[pep_combined["dataset"] == "Adult"].iterrows():
            proxy = row["proxy_variable"]
            if proxy in adult_stability:
                pep_combined.at[idx, "pc_stable_stability"] = adult_stability[proxy]["pc_stable"]
                pep_combined.at[idx, "hc_bdeu_stability"] = adult_stability[proxy]["hc_bdeu"]

        # Fill in ACS stability
        for idx, row in pep_combined[pep_combined["dataset"] == "ACSIncome"].iterrows():
            proxy = row["proxy_variable"]
            if proxy in acs_stability:
                pep_combined.at[idx, "pc_stable_stability"] = acs_stability[proxy]["pc_stable"]
                pep_combined.at[idx, "hc_bdeu_stability"] = acs_stability[proxy]["hc_bdeu"]

        print(f"✓ Added stability values")

        # ==========================================
        # 4. PRINT AND SAVE RESULTS
        # ==========================================

        print("\n" + "="*80)
        print("RESULTS TABLE")
        print("="*80)
        print(pep_combined.to_string(index=False))

        print("\n" + "="*80)
        print("INTERPRETATION")
        print("="*80)
        print("\nKey points on PEP:")
        print("• Higher PEP indicates greater target-relevant proxy information conditional on S.")
        print("• PEP is a diagnostic and should not be interpreted as a direct estimate of accuracy loss.")
        print("• PEP measures I(Y; P | S), the mutual information between target Y and proxy P")
        print("  after conditioning on the protected attribute S.")
        print("\nStability interpretation:")
        print("• PC-Stable: Fraction of PC-Stable runs (across bootstrap samples) that retained the proxy.")
        print("• HC-BDeu: Fraction of HC-BDeu runs that retained the proxy.")
        print("• High PEP + high stability: Proxy is both target-relevant and consistently discovered.")
        print("\nNote:")
        print("• PEP is an information-theoretic diagnostic, not a causal claim.")
        print("• High PEP does not prove unfairness, but indicates potential concern for fair-ML audits.")

        # Save CSV
        pep_combined.to_csv("retained_proxy_pep_values.csv", index=False)
        print("\n✓ Saved CSV: retained_proxy_pep_values.csv")

        # Save LaTeX table
        generate_latex_table(pep_combined)

        # Generate plot
        plot_pep_values(pep_combined)

        print("\n" + "="*80)
        print("✓ ANALYSIS COMPLETE")
        print("="*80)
        print("\nGenerated files:")
        print("  • retained_proxy_pep_values.csv")
        print("  • retained_proxy_pep_values.tex")
        print("  • retained_proxy_pep_values.png")
        print("="*80)

    else:
        print("\n✗ No results generated.")
        print("\nTo fix Adult dataset loading:")
        print("  1. Manual download: https://archive.ics.uci.edu/ml/datasets/adult")
        print("  2. Or use wget: wget https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data -O adult.csv")
        print("  3. Place in current directory")
        print("  4. Run again")

PROXY ENTANGLEMENT PENALTY (PEP) COMPUTATION
UCI Adult and ACSIncome Datasets

PART 1: UCI ADULT DATASET

[Loading Adult Dataset from Local File]
  Trying: adult.csv... ✗
  Trying: adult.data... ✗
  Trying: adult.txt... ✗

[Loading UCI Adult Dataset from URL]
  Trying: adult.data...   Attempting with certifi SSL certificates...
    Failed: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] ce
  Attempting with unverified SSL context...
✓
  Original shape: (32561, 15)
  After dropping NaN: (30162, 15)
  Final shape: (30162, 12)

[Preprocessing Adult for PEP computation]
  Dataset shape after preprocessing: (30162, 12)
  Computing PEP for 'relationship'... PEP = 0.0916 nats
  Computing PEP for 'marital-status'... PEP = 0.0866 nats
  Computing PEP for 'occupation'... PEP = 0.0654 nats
  Computing PEP for 'hours-per-week'... PEP = 0.0263 nats
  Computing PEP for 'education-num'... PEP = 0.0564 nats

✓ Computed PEP for 5 Adult proxies

PART 2: ACSINCOME DATASET

[Loading ACSIncome Dataset]
  D